# Seq2Seq

## Importing the libraries

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import tqdm


## For uniformity

In [5]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# Loading the data

- Data is downloaded from [here](https://huggingface.co/datasets/bentrevett/multi30k)
- Using Hugging Face datasets

In [6]:
import jsonlines
from datasets import Dataset, DatasetDict

def load_jsonl(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for obj in reader:
            data.append(obj)
    return Dataset.from_list(data)

# Load each split
train_dataset = load_jsonl("data/train.jsonl")
val_dataset = load_jsonl("data/val.jsonl")
test_dataset = load_jsonl("data/test.jsonl")

# Combine into a DatasetDict
full_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})


In [7]:
train_dataset[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

## Using `spacy` for tokenization

In [9]:
! python -m spacy download en_core_web_sm
! python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 61.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

In [11]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

In [12]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [13]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# Apply map directly to the datasets.Dataset objects from the DatasetDict
train_data = full_dataset["train"].map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = full_dataset["validation"].map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = full_dataset["test"].map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

## Building Vocabulary

In [15]:
from collections import Counter

class Vocabulary:
    def __init__(self, min_freq=1, specials=None):
        """
        Parameters:
        - min_freq: Minimum frequency a word must have to be included.
        - specials: List of special tokens like ["<unk>", "<pad>", "<sos>", "<eos>"]
        """
        self.min_freq = min_freq
        self.word_counts = Counter()
        self.specials = specials if specials else []

        self.itos = {}
        self.stoi = {}

        # Reserve space for special tokens at the beginning
        for idx, token in enumerate(self.specials):
            self.itos[idx] = token
            self.stoi[token] = idx

    def __getitem__(self, token):
        return self.stoi.get(token, self.stoi['<unk>'])

    def __len__(self):
        return len(self.itos)

    def add_sentence(self, tokens):
        self.word_counts.update(tokens)

    def build_vocabulary(self, iterator):
        """
        iterator: a list of tokenized sentences (list of list of strings)
        """
        for tokens in iterator:
            self.add_sentence(tokens)

        idx = len(self.itos)  # Start indexing after special tokens

        for word, freq in self.word_counts.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, tokens):
        unk_idx = self.stoi.get("<unk>", 0)  # fallback if <unk> not defined
        return [self.stoi.get(token, unk_idx) for token in tokens]

    def get_itos(self):
        return [self.itos[i] for i in range(len(self.itos))]

    def get_stoi(self):
        return [self.stoi[i] for i in range(len(self.stoi))]

    def lookup_tokens(self, indices):
        return [self.itos.get(index, '<unk>') for index in indices]

    def lookup_indices(self, tokens):
        return [self.stoi.get(token, self.stoi['<unk>']) for token in tokens]


In [16]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
sos_token = "<sos>"
eos_token = "<eos>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

# Create English vocab
en_vocab = Vocabulary(min_freq=min_freq, specials=special_tokens)
en_vocab.build_vocabulary(train_data["en_tokens"])  # list of tokenized English sentences

# Create German vocab
de_vocab = Vocabulary(min_freq=min_freq, specials=special_tokens)
de_vocab.build_vocabulary(train_data["de_tokens"])  # list of tokenized German sentences

In [17]:
de_vocab.get_itos()[:10]

['<unk>',
 '<pad>',
 '<sos>',
 '<eos>',
 'zwei',
 'junge',
 'weiße',
 'männer',
 'sind',
 'im']

- Ensure the special tokens of both vocabulary map to same indices

In [18]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]


In [19]:
tokens = ["i", "love", "watching", "crime", "shows"]

In [20]:
en_vocab.lookup_indices(tokens)

[171, 4010, 225, 0, 1130]

In [21]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [22]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data_numericalized = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data_numericalized = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data_numericalized = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

print(train_data_numericalized[0])
print(type(train_data_numericalized[0]["en_ids"]))


Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.', 'en_tokens': ['<sos>', 'two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<eos>'], 'de_tokens': ['<sos>', 'zwei', 'junge', 'weiße', 'männer', 'sind', 'im', 'freien', 'in', 'der', 'nähe', 'vieler', 'büsche', '.', '<eos>'], 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3], 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}
<class 'list'>


- The `with_format` method converts features indicated by the `columns` argument to a given `type`.

In [23]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

# Apply with_format to the numericalized datasets
train_data = train_data_numericalized.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data_numericalized.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data_numericalized.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

In [24]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [25]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [26]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

## Encoder

In [27]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded) # embedded = [src length, batch size, embedding dim]

        return hidden, cell


In [28]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input should be 1D: [batch_size] or 2D: [1, batch_size]
        if input.dim() == 2:
            input = input.squeeze(0)  # make input shape [batch_size]

        input = input.unsqueeze(0)  # now [1, batch_size]

        embedded = self.dropout(self.embedding(input))  # [1, batch_size, embed_dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))  # LSTM expects 3D
        prediction = self.fc_out(output.squeeze(0))  # [batch_size, output_dim]
        return prediction, hidden, cell


In [29]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = trg[0]  # shape = [1, batch_size]

        for t in range(1, trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            outputs[t] = output
            teacher_force  = random.random() < teacher_forcing_ratio

            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        return outputs


In [30]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

In [31]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [32]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


In [33]:
optimizer = optim.Adam(model.parameters())

In [34]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [35]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [36]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [37]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 10%|█         | 1/10 [00:45<06:49, 45.52s/it]

	Train Loss:   5.028 | Train PPL: 152.603
	Valid Loss:   4.873 | Valid PPL: 130.727


 20%|██        | 2/10 [01:34<06:19, 47.42s/it]

	Train Loss:   4.390 | Train PPL:  80.607
	Valid Loss:   4.656 | Valid PPL: 105.258


 30%|███       | 3/10 [02:19<05:23, 46.24s/it]

...

	Train Loss:   3.297 | Train PPL:  27.040
	Valid Loss:   4.050 | Valid PPL:  57.412


 90%|█████████ | 9/10 [06:48<00:44, 44.84s/it]

	Train Loss:   3.166 | Train PPL:  23.702
	Valid Loss:   3.950 | Valid PPL:  51.953


100%|██████████| 10/10 [07:32<00:00, 45.28s/it]

	Train Loss:   3.048 | Train PPL:  21.064
	Valid Loss:   3.867 | Valid PPL:  47.808


In [38]:
model.load_state_dict(torch.load("tut1-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

| Test Loss: 3.846 | Test PPL:  46.804 |


In [39]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [40]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [41]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [42]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'black',
 'hat',
 'is',
 'cutting',
 'food',
 '.',
 '<eos>']

In [43]:
sentence = "Ein Mann sitzt auf einer Bank."

In [44]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [45]:
translation

['<sos>', 'a', 'man', 'sitting', 'on', 'a', 'bench', '.', '<eos>']

In [46]:
translations = [
    translate_sentence(
        example["de"],
        model,
        en_nlp,
        de_nlp,
        en_vocab,
        de_vocab,
        lower,
        sos_token,
        eos_token,
        device,
    )
    for example in tqdm.tqdm(test_data)
]

100%|██████████| 1000/1000 [00:08<00:00, 118.97it/s]


In [48]:
import evaluate

In [49]:
bleu = evaluate.load("bleu")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [50]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [51]:
predictions[0], references[0]

('a man in a black hat is cutting food .',
 ['A man in an orange hat starring at something.'])

In [52]:
def get_tokenizer_fn(nlp, lower):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lower:
            tokens = [token.lower() for token in tokens]
        return tokens

    return tokenizer_fn

In [53]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lower)

In [54]:
tokenizer_fn(predictions[0]), tokenizer_fn(references[0][0])

(['a', 'man', 'in', 'a', 'black', 'hat', 'is', 'cutting', 'food', '.'],
 ['a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.'])

In [55]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

In [56]:
results

{'bleu': 0.1323227862965373,
 'precisions': [0.4748099976494555,
  0.18660205729830825,
  0.08928737340890086,
  0.04250742599610775],
 'brevity_penalty': 0.9771513870670351,
 'length_ratio': 0.9774084852197886,
 'translation_length': 12763,
 'reference_length': 13058}

In [57]:
import torch
import torch.nn.functional as F

def beam_search_translate(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    beam_width=5,
    max_output_length=25
):
    model.eval()

    with torch.no_grad():
        # Preprocess sentence
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]

        if lower:
            tokens = [token.lower() for token in tokens]

        tokens = [sos_token] + tokens + [eos_token]

        src_tensor = torch.LongTensor(de_vocab.lookup_indices(tokens)).unsqueeze(1).to(device)

        # Get encoder hidden state
        hidden, cell = model.encoder(src_tensor)

        # Initialize beam with (score, sequence, hidden, cell)
        sos_idx = en_vocab[sos_token]
        eos_idx = en_vocab[eos_token]

        beam = [(0.0, [sos_idx], hidden, cell)]
        completed_sequences = []

        for _ in range(max_output_length):
            new_beam = []

            for score, seq, h, c in beam:
                last_token = torch.LongTensor([seq[-1]]).to(device)

                if seq[-1] == eos_idx:
                    completed_sequences.append((score, seq))
                    continue

                output, new_h, new_c = model.decoder(last_token, h, c)
                log_probs = F.log_softmax(output, dim=1)
                top_log_probs, top_indices = log_probs.topk(beam_width)

                for i in range(beam_width):
                    token = top_indices[0][i].item()
                    token_log_prob = top_log_probs[0][i].item()
                    new_seq = seq + [token]
                    new_score = score - token_log_prob  # negate to use min() later
                    new_beam.append((new_score, new_seq, new_h, new_c))

            # Keep best beam_width sequences
            beam = sorted(new_beam, key=lambda x: x[0])[:beam_width]

        # Add any remaining sequences that hit <eos>
        for score, seq, _, _ in beam:
            if seq[-1] == eos_idx:
                completed_sequences.append((score, seq))

        # If no completed sequences, fall back to the current best
        if not completed_sequences:
            completed_sequences = [(score, seq) for score, seq, _, _ in beam]

        best_seq = min(completed_sequences, key=lambda x: x[0])[1]

        return en_vocab.lookup_tokens(best_seq)


In [58]:
translated_tokens = beam_search_translate(
    sentence="ein kleines mädchen klettert in ein spielhaus .",
    model=model,
    en_nlp=en_nlp,
    de_nlp=de_nlp,
    en_vocab=en_vocab,
    de_vocab=de_vocab,
    lower=True,
    sos_token="<sos>",
    eos_token="<eos>",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    beam_width=5,
    max_output_length=25
)

print(" ".join(translated_tokens))


<sos> a little girl is a a . . <eos>


In [59]:
! pip install nltk


In [60]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


In [61]:
# Reference translation (ground truth)
reference = [["a", "little", "girl", "is", "climbing", "into", "a", "playhouse", "."]]  # List of list of tokens

# Hypothesis (your model's prediction)
hypothesis = ["a", "young", "girl", "climbs", "into", "a", "playhouse", "."]

# Optional: smoothing function helps with short sentences
smooth = SmoothingFunction().method4

# Compute BLEU-1 to BLEU-4 (cumulative)
bleu_score = sentence_bleu(reference, hypothesis, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
print(f"BLEU score: {bleu_score:.4f}")


BLEU score: 0.3376


In [62]:
print(test_data[0])


{'en_ids': tensor([  2,  21,  31,  17, 202,  96, 152, 690,  40, 178,  14,   3]), 'de_ids': tensor([  2,  21,  28,  18,  29,  97, 200,  49,  12, 174,   0,  16,   3]), 'en': 'A man in an orange hat starring at something.', 'de': 'Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.', 'en_tokens': ['<sos>', 'a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.', '<eos>'], 'de_tokens': ['<sos>', 'ein', 'mann', 'mit', 'einem', 'orangefarbenen', 'hut', ',', 'der', 'etwas', 'anstarrt', '.', '<eos>']}


In [63]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

total_bleu = 0
num_sentences = len(test_data)

for item in test_data:
    src_sentence = item["de"]
    tgt_sentence = item["en"]

    predicted = beam_search_translate(
        sentence=src_sentence,
        model=model,
        en_nlp=en_nlp,
        de_nlp=de_nlp,
        en_vocab=en_vocab,
        de_vocab=de_vocab,
        lower=True,
        sos_token="<sos>",
        eos_token="<eos>",
        device=device,
        beam_width=5
    )

    # Clean up predicted tokens
    predicted = [tok for tok in predicted if tok not in ["<sos>", "<eos>", "<pad>"]]

    # Reference sentence as list of tokens
    reference = [[token.text.lower() for token in en_nlp.tokenizer(tgt_sentence)]]

    score = sentence_bleu(reference, predicted, weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=SmoothingFunction().method4)

    total_bleu += score

average_bleu = total_bleu / num_sentences
print(f"\nAverage BLEU score on test set: {average_bleu:.4f}")



Average BLEU score on test set: 0.1417


In [64]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

total_bleu = 0
num_sentences = len(test_data)
smooth_fn = SmoothingFunction().method4

print("\nSample Translations:\n")

# Limit number of printed examples
num_print = 5
printed = 0

for item in test_data:
    src_sentence = item["de"]
    tgt_sentence = item["en"]

    predicted = beam_search_translate(
        sentence=src_sentence,
        model=model,
        en_nlp=en_nlp,
        de_nlp=de_nlp,
        en_vocab=en_vocab,
        de_vocab=de_vocab,
        lower=True,
        sos_token="<sos>",
        eos_token="<eos>",
        device=device,
        beam_width=5
    )

    # Clean predicted tokens
    predicted = [tok for tok in predicted if tok not in ["<sos>", "<eos>", "<pad>"]]

    # Tokenize reference
    reference = [[token.text.lower() for token in en_nlp.tokenizer(tgt_sentence)]]

    # Compute sentence BLEU
    score = sentence_bleu(reference, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
    total_bleu += score

    # Print a few example translations
    if printed < num_print:
        print(f"German (src):     {src_sentence}")
        print(f"Reference (gold): {tgt_sentence}")
        print(f"Predicted (beam): {' '.join(predicted)}")
        print(f"BLEU score:       {score:.4f}")
        print("-" * 60)
        printed += 1

# Final average BLEU
average_bleu = total_bleu / num_sentences
print(f"\nAverage BLEU score on test set: {average_bleu:.4f}")




Sample Translations:

German (src):     Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Reference (gold): A man in an orange hat starring at something.
Predicted (beam): a man in an orange vest is cutting food .
BLEU score:       0.4111
------------------------------------------------------------
German (src):     Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
Reference (gold): A Boston Terrier is running on lush green grass in front of a white fence.
Predicted (beam): a black dog runs through the grass near a green dog .
BLEU score:       0.0215
------------------------------------------------------------
German (src):     Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
Reference (gold): A girl in karate uniform breaking a stick with a front kick.
Predicted (beam): a girl in an orange outfit is a a a a .
BLEU score:       0.1158
------------------------------------------------------------
German (src):     Fünf Leute in Winterjac